# ComicAnalizer - Magi + PaddleOCR Pipeline

Este notebook genera la salida base para trabajar: detecciones Magi, reporte de calidad y comparacion OCR complementaria con PaddleOCR.

Antes de correrlo, activa GPU en Colab: `Runtime > Change runtime type > T4 GPU`.

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!rm -rf /content/ComicAnalizer
!git clone https://github.com/nicolas4432/ComicAnalizer.git /content/ComicAnalizer
%cd /content/ComicAnalizer
!git log --oneline -5

In [ ]:
%cd /content/ComicAnalizer
!pip -q install \
  "transformers==4.49.0" \
  "huggingface_hub<1.0" \
  timm \
  einops \
  pytorch-metric-learning \
  shapely \
  "paddleocr==3.3.3" \
  "paddlepaddle==3.2.0"

## Subir dataset limpio

Para probar el c?mic nuevo sube `magi_nekkorarekko_clean.zip`, generado localmente desde `outputs/packages/magi_nekkorarekko_clean.zip`.

El notebook tambi?n sirve para `magi_clean_full.zip`: cambia `PACKAGE_NAME`, `RUN_NAME` y deja `COMIC_ID = ''` si quieres todos los c?mics.


In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
PACKAGE_NAME = zip_name.replace('.zip', '')
print('ZIP subido:', zip_name)
print('PACKAGE_NAME detectado:', PACKAGE_NAME)

!rm -rf /content/magi_sample
!mkdir -p /content/magi_sample
!unzip -q -o "$zip_name" -d /content/magi_sample
!find /content/magi_sample -maxdepth 5 -type d | head -40


## Ejecutar Magi

`COMIC_ID = 'nekkorarekko'` procesa solo ese c?mic. Si quieres todos los c?mics del ZIP, usa `COMIC_ID = ''` y `MAX_COMICS = 0`.


In [ ]:
RUN_NAME = 'nekkorarekko_clean'
DATASET_NAME = 'test_1_clean'
COMIC_ID = 'nekkorarekko'  # usa '' para no filtrar por comic
MAX_COMICS = 0

RUN_ROOT = f'outputs/runs/{RUN_NAME}'
MAGI_OUTPUT = f'{RUN_ROOT}/magi'
MAGI_VISUALS = f'{RUN_ROOT}/visuals/magi_boxes'
OCR_VISUALS = f'{RUN_ROOT}/visuals/ocr_boxes'
ANALYSIS_OUTPUT = f'{RUN_ROOT}/analysis/magi_analysis_report.json'
OCR_OUTPUT = f'{RUN_ROOT}/analysis/paddle_magi_ocr_comparison.json'
MAGI_CACHE = 'outputs/cache/magi'
IMAGE_ROOT = f'/content/magi_sample/{PACKAGE_NAME}/by_comic'

comic_filter = f'--comic-id {COMIC_ID}' if COMIC_ID else ''

!python -m tools.inspect_magi_dataset   --input $IMAGE_ROOT   --output-dir $MAGI_OUTPUT   --visual-output-dir $MAGI_VISUALS   --all-pages-per-comic   --no-panel-crops   --dataset-name $DATASET_NAME   --task detections   --cache-dir $MAGI_CACHE   --device cuda   --dtype float16   --max-comics $MAX_COMICS   $comic_filter


## Generar reporte normalizado de calidad

In [ ]:
!python -m tools.analyze_magi_results \
  --input $MAGI_OUTPUT \
  --output $ANALYSIS_OUTPUT \
  --top-n 20

In [ ]:
import json
from pathlib import Path

report = json.loads(Path(ANALYSIS_OUTPUT).read_text())
print(json.dumps(report['summary'], indent=2, ensure_ascii=False))
print('\nPaginas sospechosas:', report['summary']['suspicious_page_count'])
print('\nTop flags:', report['summary']['flag_counts'])

## Comparar Magi contra PaddleOCR

Corre una muestra aleatoria primero. Sube `OCR_LIMIT` despues si el tiempo es razonable.

In [ ]:
OCR_LIMIT = 12
ocr_comic_filter = f'--comic-id {COMIC_ID}' if COMIC_ID else ''

!python -m tools.compare_magi_paddleocr   --magi-input $MAGI_OUTPUT   --image-root $IMAGE_ROOT   --dataset-name $DATASET_NAME   --selection random   --limit $OCR_LIMIT   --seed 42   --lang en   --visual-output-dir $OCR_VISUALS   --output $OCR_OUTPUT   $ocr_comic_filter


In [ ]:
ocr_report = json.loads(Path(OCR_OUTPUT).read_text())
print(json.dumps(ocr_report['summary'], indent=2, ensure_ascii=False))
for item in ocr_report['comparisons'][:10]:
    print(item['comic_id'], item['file_name'], 'Magi=', item['magi_text_regions'], 'Paddle=', item['paddle_text_blocks'], 'match=', item['matched_regions'], 't=', round(item['paddle_elapsed_seconds'], 2))

## Descargar salida estandar

In [ ]:
from google.colab import files

zip_out = f'{RUN_NAME}_magi_ocr_outputs.zip'
!zip -qr "$zip_out"   $RUN_ROOT   $MAGI_CACHE

files.download(zip_out)
